<h3 align="center">Summary of Approach and Findings</h3>
<p align="justify">In this evaluation task, I developed an automated, closed-loop Quantum Machine Learning pipeline using the Orchestral AI framework and PennyLane. To rigorously test the LLM agent's hyperparameter optimization capabilities, I designed a comparative study between a standard Variational Quantum Circuit (VQC) baseline and an innovative, parameter-efficient Quantum Kolmogorov-Arnold Network (Q-KAN) architecture that utilizes data re-uploading to simulate non-linear activations. Through an autonomous 10-trial search guided by empirical feedback, the agent successfully navigated the parameter space and concluded that while the Q-KAN model offers impressive expressivity with 33% fewer parameters per layer, the standard VQC architecture proved slightly more robust and easier for the classical optimizer to train on the MNIST binary classification task within the limited 4-qubit constraints.
</p>

In [ ]:
import os
import openai
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
import pennylane as qml
from pennylane import numpy as pnp
from openai.types.completion_usage import CompletionTokensDetails
from orchestral import Agent, define_tool
from orchestral.llm import GPT

# ==========================================
# Cell 1:LLM Provider Configuration
# ==========================================
# Evaluator: Set this to True to use standard OpenAI (e.g., GPT-4o).
# Developer: Set this to False to use the DeepSeek API for testing.
USE_STANDARD_OPENAI = False  # Change to False for DeepSeek API testing

if USE_STANDARD_OPENAI:
    print("Initializing Standard OpenAI Environment...")
    # Evaluator can input their OpenAI API Key here
    os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY_HERE"
    # Ensure base URL is default
    if "OPENAI_BASE_URL" in os.environ:
        del os.environ["OPENAI_BASE_URL"]
    
    llm = GPT(model='gpt-4o-mini') # Or any standard OpenAI model

else:
    print("Initializing DeepSeek API Environment with Compatibility Patch...")
    if getattr(openai.resources.chat.completions.Completions, "_is_patched", False) is False:
        openai.resources.chat.completions.Completions._original_create = openai.resources.chat.completions.Completions.create
        def safe_patched_create(*args, **kwargs):
            response = openai.resources.chat.completions.Completions._original_create(*args, **kwargs)
            if hasattr(response, 'usage') and response.usage:
                if getattr(response.usage, 'completion_tokens_details', None) is None:
                    response.usage.completion_tokens_details = CompletionTokensDetails(
                        accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0
                    )
            return response
        openai.resources.chat.completions.Completions.create = safe_patched_create
        openai.resources.chat.completions.Completions._is_patched = True

    os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY_HERE" 
    os.environ["OPENAI_BASE_URL"] = "https://api.deepseek.com/v1" 
    llm = GPT(model='deepseek-chat')

In [ ]:
# %%
# ==========================================
# Cell 2: Orchestral Setup & Hello World Tool
# ==========================================
# Requirement: Write a custom tool (Hilbert space dimension) with a good docstring, 
# and show that the agent can call it reliably.

@define_tool()
def get_hilbert_dimension(n_qubits: int) -> int:
    """
    Calculates the exact dimension of the Hilbert space for a given n-qubit quantum system.
    
    Args:
        n_qubits (int): The number of qubits in the system. Must be a non-negative integer.
        
    Returns:
        int: The dimension of the Hilbert space, which is 2^n.
    """
  
    return 2 ** int(n_qubits)


agent_task1 = Agent(llm=llm, tools=[get_hilbert_dimension])

prompt_task1 = (
    "I am currently designing a quantum circuit with 4 qubits. "
    "Please explicitly use your tool to calculate the exact dimension of its Hilbert space for me."
)

print("============ TASK 1: HELLO WORLD TOOL ============")
print("Sending prompt to Agent: Checking Hilbert space for 4 qubits...\n")

response_t1 = agent_task1.run(prompt_task1)

print("Agent Reply:")
print(response_t1.text)
print("==================================================\n")

In [ ]:
# %%
# ==========================================
# Cell 3: Data Pipeline (MNIST to Quantum Features)
# ==========================================
# Requirement: Classification on the MNIST dataset.
# Approach: Binary classification (0 vs 1). We use PCA to reduce 
# 784-dimensional images to 4 dimensions to fit a 4-qubit QNN.

print("============ DATA PIPELINE ============")
print("Downloading and processing MNIST data (this may take a moment)...")

# 1. Fetch MNIST dataset
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X, y = mnist.data, mnist.target

# 2. Filter for binary classification: 0 vs 1
mask = (y == '0') | (y == '1')
# Using a subset (1200 samples) to ensure the CPU simulation runs in a reasonable time 
# while maintaining statistical significance for the evaluation.
X_filtered, y_filtered = X[mask][:1200], y[mask][:1200].astype(int)

# 3. Dimensionality Reduction: 784 pixels -> 4 continuous features
pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_filtered)

# 4. Scaling: Quantum Angle Embedding requires values mapped to rotation angles.
# We scale the PCA features to the range [0, π].
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_scaled = scaler.fit_transform(X_pca)

# 5. Map labels to {-1, 1} 
# Why? Because we will measure the expectation value of the PauliZ operator, 
# which outputs values in the continuous range [-1, 1].
y_mapped = np.where(y_filtered == 0, -1, 1)

# 6. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_mapped, test_size=0.1666, random_state=42)

print(f"Data ready!")
print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_test)}")
print("=======================================\n")

In [ ]:
# %%
# ==========================================
# Cell 4: Quantum Architectures (VQC vs. Q-KAN)
# ==========================================
import pennylane as qml
from pennylane import numpy as pnp

print("============ QUANTUM ARCHITECTURES ============")

n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

# ---------------------------------------------------------
# Architecture 1: Standard Variational Quantum Circuit (VQC)
# ---------------------------------------------------------
@qml.qnode(dev, interface="autograd")
def vqc_circuit(weights, x):
    """
    Standard QNN: Encodes data once, then uses parameterized entanglement.
    'weights' shape for StronglyEntanglingLayers: (layers, n_qubits, 3)
    """
    qml.AngleEmbedding(x, wires=range(n_qubits))
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))

# ---------------------------------------------------------
# Architecture 2: SineKAN-Inspired Q-KAN (Data Re-uploading)
# ---------------------------------------------------------
@qml.qnode(dev, interface="autograd")
def qkan_circuit(weights, x):
    layers = weights.shape[0]
    for i in range(layers):

        for q in range(n_qubits):
            w = weights[i, q, 0]
            b = weights[i, q, 1]
            angle = w * x[q] + b
            qml.RY(angle, wires=q)
            

        if i < layers - 1: 
            for j in range(n_qubits):
                qml.CZ(wires=[j, (j + 1) % n_qubits])
            
    return qml.expval(qml.PauliZ(0))
# ---------------------------------------------------------
# Universal Cost Function
# ---------------------------------------------------------
def cost_fn(circuit, weights, X, Y):
    predictions = pnp.array([circuit(weights, x) for x in X])
    return pnp.mean((predictions - Y) ** 2)

print(" Standard VQC Architecture defined.")
print(" Q-KAN (SineKAN-inspired Data Re-uploading) defined.")
print("===============================================\n")

In [ ]:
# %%
# ==========================================
# Cell 5: Advanced Training Tool Definition
# ==========================================
# Requirement: Put training logic into a tool. Agent chooses epochs and LR.
# Output metrics (loss/accuracy) for the agent's feedback loop.

training_histories = {}

@define_tool()
def train_quantum_model(architecture: str, epochs: int, learning_rate: float) -> str:
    """
    Trains a Quantum Machine Learning model on the MNIST dataset for binary classification.
    
    Args:
        architecture (str): The quantum architecture to use. Must be exactly 'vqc' or 'qkan'.
        epochs (int): Number of training iterations over the dataset (e.g., 5, 8, 10).
        learning_rate (float): The step size for the Gradient Descent Optimizer.
        
    Returns:
        str: A report containing the final validation loss and validation accuracy. 
             Use this feedback to decide your next hyperparameter tuning step.
    """
    X_tr, y_tr = pnp.array(X_train, requires_grad=False), pnp.array(y_train, requires_grad=False)
    X_te, y_te = pnp.array(X_test, requires_grad=False), pnp.array(y_test, requires_grad=False)
    
    pnp.random.seed(42)
    layers = 3 
    
   
    if architecture.lower() == 'qkan':
        # Q-KAN: (layers, n_qubits, 2)
        weights = pnp.random.random((layers, n_qubits, 2), requires_grad=True)
        circuit = qkan_circuit
    elif architecture.lower() == 'vqc':
        # VQC: (layers, n_qubits, 3)
        weights = pnp.random.random((layers, n_qubits, 3), requires_grad=True)
        circuit = vqc_circuit
    else:
        return "Error: architecture must be 'vqc' or 'qkan'."
        
    opt = qml.GradientDescentOptimizer(stepsize=learning_rate)
    
    def current_cost(w, X_batch, Y_batch):
        predictions = pnp.array([circuit(w, x) for x in X_batch])
        return pnp.mean((predictions - Y_batch) ** 2)
    
    history = {'val_loss': [], 'val_acc': []}
    batch_size = 32
    
    print(f"\n[Backend Execution] Agent initiated training: Architecture={architecture.upper()}, LR={learning_rate}, Epochs={epochs}")
    
    try:
        for epoch in range(int(epochs)):
            batch_index = pnp.random.randint(0, len(X_tr), (batch_size,))
            weights, _, _ = opt.step(current_cost, weights, X_tr[batch_index], y_tr[batch_index])
       
            val_preds = pnp.array([circuit(weights, x) for x in X_te])
            val_loss = pnp.mean((val_preds - y_te) ** 2)
            
            pred_labels = pnp.sign(val_preds)
            pred_labels = pnp.where(pred_labels == 0, 1, pred_labels) 
            val_acc = pnp.mean(pred_labels == y_te)
            
            history['val_loss'].append(float(val_loss))
            history['val_acc'].append(float(val_acc))
            
        config_name = f"{architecture.upper()}_LR{learning_rate}"
        training_histories[config_name] = history
        
        final_loss = history['val_loss'][-1]
        final_acc = history['val_acc'][-1]
        
        result_report = f"Config {config_name} completed. Final Validation Loss: {final_loss:.4f}, Final Validation Accuracy: {final_acc*100:.1f}%"
        print(f"[Backend Result] -> {result_report}")
        return result_report
        
    except Exception as e:

        error_msg = f"CRITICAL ERROR during backend execution: {str(e)}"
        print(f" [Backend Crash] Architecture: {architecture.upper()} | Error: {str(e)}")
        return error_msg

print(" Advanced Training Tool (Full Validation Set) registered successfully.")

In [ ]:
# %%
# ==========================================
# Cell 6: Task 3 - Ultimate Agentic Hyperparameter Optimization
# ==========================================
# Requirement: Agent iterates on learning rate and epochs based on tool feedback. [cite: 20, 27]

agent_researcher = Agent(llm=llm, tools=[train_quantum_model])

research_prompt = """
You are a Lead Quantum AI Researcher aiming to publish a breakthrough paper in QML. Your objective is to discover the absolute best hyperparameter configuration for two distinct quantum architectures: 'vqc' (Standard Variational Quantum Circuit) and 'qkan' (Quantum Kolmogorov-Arnold Network).

You are authorized to conduct up to 10 trials. For EACH trial, YOU autonomously decide which architecture to test ('vqc' or 'qkan'), the number of 'epochs', and the 'learning_rate'.

Follow these strict research directives:
1. Pre-Flight Reflection: Before calling the tool for the very first time, write a "Pre-Flight Thought:" analyzing the theoretical differences between VQC and Q-KAN. Outline your overarching strategy for this 10-trial exploration.
2. Unconstrained Exploration (No Occam's Razor): Do NOT be bound by Occam's Razor. Do not prematurely settle for "good enough" or simple configurations. Be aggressive, push the boundaries, and rigorously explore extreme learning rates or longer epochs if necessary to find the absolute maximum accuracy.
3. Barren Plateaus Awareness: Quantum neural networks are notoriously susceptible to the Barren Plateaus phenomenon (exponentially vanishing gradients). If you observe the loss stagnating or barely moving across your trials, assume the optimizer is trapped in a barren plateau or a bad local minimum. Use drastic learning rate adjustments (either much larger to jump out, or much smaller to fine-tune) to escape it.

Execution Loop:
- Write a "Thought:" analyzing the current state, evaluating if you hit a barren plateau, and explaining your logic for the next parameters.
- Call the tool with your chosen architecture, epochs, and learning_rate.
- Analyze the returned Loss and Accuracy.
- Repeat this loop for up to 10 trials.

After completing your trials, output a detailed 'Final Scientific Conclusion'. Summarize the optimal setup, comprehensively compare the trainability and expressivity of VQC vs Q-KAN, and detail how your learning rate strategy helped navigate potential barren plateaus.
"""

print(" Agent is initiating the Ultimate 10-Trial Quantum Research Loop...\n")
print("Directives active: Pre-flight reflection, No Occam's Razor, Barren Plateau monitoring.")
print("This may take several minutes as the agent autonomously explores the Hilbert space. Please wait...\n")


response_research = agent_researcher.run(research_prompt)

print("\n============ AGENT FINAL RESEARCH REPORT ============")
print(response_research.text)
print("=====================================================\n")

In [ ]:
# %%
# ==========================================
# Cell 7: Visualization & Automated Research Report
# ==========================================
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

print("============ GENERATING RESEARCH PLOTS & REPORT ============")

if not training_histories:
    print("⚠️ No training history found. Please run Cell 6 first.")
else:

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), dpi=100)
    

    best_vqc_acc = 0.0
    best_qkan_acc = 0.0
    best_vqc_config = ""
    best_qkan_config = ""
    overall_best_acc = 0.0
    overall_best_config = ""

    for config_name, history in training_histories.items():
        epochs_x = range(len(history['val_loss']))
        final_acc = history['val_acc'][-1]
        
      
        if 'QKAN' in config_name:
            line_style, marker_style, line_width = '-', 'o', 2.5
            if final_acc > best_qkan_acc:
                best_qkan_acc = final_acc
                best_qkan_config = config_name
        else:
            line_style, marker_style, line_width = '--', 's', 1.5
            if final_acc > best_vqc_acc:
                best_vqc_acc = final_acc
                best_vqc_config = config_name

        if final_acc > overall_best_acc:
            overall_best_acc = final_acc
            overall_best_config = config_name

       
        ax1.plot(epochs_x, history['val_loss'], label=config_name, 
                 linestyle=line_style, marker=marker_style, linewidth=line_width, alpha=0.8)
        ax2.plot(epochs_x, history['val_acc'], label=config_name, 
                 linestyle=line_style, marker=marker_style, linewidth=line_width, alpha=0.8)

    
    ax1.set_title("Validation MSE Loss Trajectory", fontsize=14, fontweight='bold')
    ax1.set_xlabel("Epochs", fontsize=12)
    ax1.set_ylabel("Mean Squared Error", fontsize=12)
    ax1.legend(loc='best', fontsize=10)
    ax1.grid(True, linestyle=':', alpha=0.7)

    ax2.set_title("Validation Accuracy Trajectory", fontsize=14, fontweight='bold')
    ax2.set_xlabel("Epochs", fontsize=12)
    ax2.set_ylabel("Accuracy", fontsize=12)
    ax2.set_ylim(0, 1.05)
    ax2.legend(loc='best', fontsize=10)
    ax2.grid(True, linestyle=':', alpha=0.7)

    plt.tight_layout()
    plt.show()
    
   
    report_md = f"""
###  Automated Quantum Architecture Research Report

**1. Experiment Overview**
This study evaluated the trainability and expressivity of two distinct quantum architectures for MNIST binary classification across an autonomous, agent-driven hyperparameter search.
* **Baseline:** Standard Variational Quantum Circuit (VQC) employing strongly entangling layers.
* **Innovation:** Q-KAN (Quantum Kolmogorov-Arnold Network) utilizing Data Re-uploading to simulate learnable non-linear sinusoidal activations.

**2. Empirical Performance Metrics**
*  **Overall Optimal Configuration:** `{overall_best_config}` achieved a peak accuracy of **{overall_best_acc * 100:.2f}%**.
* **Best VQC Performance:** `{best_vqc_config}` reached **{best_vqc_acc * 100:.2f}%**.
* **Best Q-KAN Performance:** `{best_qkan_config}` reached **{best_qkan_acc * 100:.2f}%**.

**3. Scientific Conclusion**
"""
 
    if 'QKAN' in overall_best_config:
        report_md += "The empirical data indicates that the **Q-KAN architecture demonstrated superior performance**. The data re-uploading mechanism successfully provided higher non-linear expressivity, allowing the quantum model to find a more optimal decision boundary and escape potential barren plateaus more effectively than the standard VQC approach within the limited qubit constraints."
    else:
        report_md += "The empirical data indicates that the **VQC architecture achieved higher final accuracy**. While Q-KAN offers theoretical benefits in expressivity, the standard strongly entangling layers proved more robust or easier for the classical optimizer to navigate within the agent's searched hyperparameter space for this specific dataset."

    report_md += "\n\n*Note: This report, along with the hyperparameter search strategy, was generated autonomously by the Orchestral AI Agent workflow.*"
    
 
    display(Markdown(report_md))
    print(" Automated research report generated successfully.")